In [1]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf

from tensorflow import keras
from keras.models import Model
from keras.layers import Input, Dense
from keras.callbacks import EarlyStopping

from sklearn.metrics import classification_report, confusion_matrix

In [2]:
caminho_pasta_tratado = '../../dataset tratado/lycos-cicids2017/Sem Redução de Dimensionalidade/'

nome_dados_treinamento = 'lycos_cicids2017_treinamento.csv'
nome_dados_teste       = 'lycos_cicids2017_teste.csv'

In [3]:
print("Carregando dataset de treinamento...")
df_treino_full = pd.read_csv(caminho_pasta_tratado + nome_dados_treinamento)
print(f"Dataset completo: {df_treino_full.shape}")

df_treino_benign = df_treino_full[df_treino_full['Label'] == 'benign'].copy()
print(f"Apenas BENIGN: {df_treino_benign.shape}")

X_treino = df_treino_benign.drop('Label', axis=1).values

display(df_treino_benign.head())

Carregando dataset de treinamento...
Dataset completo: (1286248, 80)
Apenas BENIGN: (977036, 80)


,src_port,dst_port,ip_prot,timestamp,flow_duration,down_up_ratio,pkt_len_max,pkt_len_min,pkt_len_mean,pkt_len_var,...,bwd_bulk_bytes_mean,bwd_bulk_pkt_mean,bwd_bulk_rate_mean,fwd_subflow_bytes_mean,fwd_subflow_pkt_mean,bwd_subflow_bytes_mean,bwd_subflow_pkt_mean,fwd_tcp_init_win_bytes,bwd_tcp_init_win_bytes,Label
0,0.951110,0.000809,0.122137,0.493151,0.000504,0.123909,0.003948,0.031768,0.030108,0.000050,...,0.000000,0.000000,0.000000,0.000026,0.000005,1.495151e-07,0.000003,0.000000,0.000000,benign
1,0.766598,0.006760,0.038168,0.235429,0.025492,0.041303,0.001249,0.000000,0.003345,0.000009,...,0.000000,0.000000,0.000000,0.000009,0.000007,0.000000e+00,0.000002,0.003922,0.021423,benign
2,0.814466,0.006760,0.038168,0.512447,0.979619,0.111518,0.057131,0.000000,0.066731,0.004397,...,0.000004,0.000069,0.000009,0.000228,0.000030,2.463439e-06,0.000021,0.445572,0.647110,benign
3,0.074983,0.000809,0.122137,0.240826,0.000002,0.123909,0.010475,0.022790,0.061262,0.000639,...,0.000000,0.000000,0.000000,0.000037,0.000009,7.933453e-07,0.000007,0.000000,0.000000,benign
4,0.847898,0.006760,0.038168,0.235855,0.070290,0.149890,0.175020,0.000000,0.392759,0.041693,...,0.000387,0.000566,0.000418,0.000322,0.000094,6.456254e-05,0.000086,0.445572,0.176773,benign


In [4]:
# Arquitetura do Autoencoder
# Encoder: n → 64 → 32 → 16 | Decoder: 16 → 32 → 64 → n
n_features = X_treino.shape[1]

inputs = Input(shape=(n_features,))

# Encoder
encoded = Dense(64, activation='relu')(inputs)
encoded = Dense(32, activation='relu')(encoded)
encoded = Dense(16, activation='relu')(encoded)

# Decoder
decoded = Dense(32, activation='relu')(encoded)
decoded = Dense(64, activation='relu')(decoded)
outputs = Dense(n_features, activation='sigmoid')(decoded)

autoencoder = Model(inputs, outputs)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 79)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 79)             │         5,135 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,519 (60.62 KB)

 Trainable params: 15,519 (60.62 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Treinamento do Autoencoder (apenas em tráfego BENIGN)
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("Iniciando treinamento do Autoencoder...")
inicio_treino = time.time()

history = autoencoder.fit(
    X_treino, X_treino,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

fim_treino = time.time()
print(f"\nTreinamento concluído! Tempo total: {fim_treino - inicio_treino:.2f} segundos.")

Iniciando treinamento do Autoencoder...
Epoch 1/50
3435/3435 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 0.0036 - val_loss: 2.5615e-04
Epoch 2/50
3435/3435 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - loss: 1.8706e-04 - val_loss: 1.4628e-04
Epoch 3/50
3435/3435 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 1.2160e-04 - val_loss: 1.0664e-04
Epoch 4/50
3435/3435 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - loss: 8.8135e-05 - val_loss: 8.3756e-05
Epoch 5/50
3435/3435 ━━━━━━━━━━━━━━━━━━━━ 3s 990us/step - loss: 7.2639e-05 - val_loss: 7.4696e-05
Epoch 6/50
3435/3435 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - loss: 6.4011e-05 - val_loss: 7.5042e-05
Epoch 7/50
3435/3435 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 5.8717e-05 - val_loss: 6.0896e-05
Epoch 8/50
3435/3435 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 5.3803e-05 - val_loss: 6.2523e-05
Epoch 9/50
3435/3435 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - loss: 5.0386e-05 - val_loss: 5.3154e-05
Epoch 10/50
3435/3435 ━━━━━━━━━━━━━━━━━━━━ 3s 998us/step - loss: 4.7278e-05 - val_loss: 5.2414e-05

In [6]:
# Definindo o limiar de anomalia (percentil 95 do erro de reconstrução no treino BENIGN)
X_treino_rec = autoencoder.predict(X_treino, verbose=0)
erros_treino = np.mean(np.power(X_treino - X_treino_rec, 2), axis=1)

PERCENTIL = 95
limiar = np.percentile(erros_treino, PERCENTIL)

print(f"Estatísticas do erro de reconstrução (BENIGN treino):")
print(f"  Média:   {erros_treino.mean():.6f}")
print(f"  Desvio:  {erros_treino.std():.6f}")
print(f"  Máximo:  {erros_treino.max():.6f}")
print(f"  Limiar ({PERCENTIL}º percentil): {limiar:.6f}")

Estatísticas do erro de reconstrução (BENIGN treino):
  Média:   0.000018
  Desvio:  0.000473
  Máximo:  0.118517
  Limiar (95º percentil): 0.000053


In [7]:
def avaliar_autoencoder(autoencoder, limiar, df_teste, nome_cenario, benign_label):
    X_teste = df_teste.drop('Label', axis=1).values
    y_teste_real = df_teste['Label'].values

    X_teste_rec = autoencoder.predict(X_teste, verbose=0)
    erros_teste = np.mean(np.power(X_teste - X_teste_rec, 2), axis=1)

    y_pred_binario = np.where(erros_teste > limiar, 'ATTACK', 'BENIGN')
    y_real_binario = np.where(y_teste_real == benign_label, 'BENIGN', 'ATTACK')

    labels_bin = ['BENIGN', 'ATTACK']
    cm = confusion_matrix(y_real_binario, y_pred_binario, labels=labels_bin)
    cm_df = pd.DataFrame(
        cm,
        index=[f"Real_{l}" for l in labels_bin],
        columns=[f"Pred_{l}" for l in labels_bin]
    )

    print(f"\n{'='*60}")
    print(f"CENÁRIO: {nome_cenario}")
    print(f"Total de amostras: {len(y_teste_real)}")
    print(f"  BENIGN: {(y_real_binario == 'BENIGN').sum()}")
    print(f"  ATTACK: {(y_real_binario == 'ATTACK').sum()}")
    print(f"\n=== MATRIZ DE CONFUSÃO ===")
    display(cm_df.style.format("{:.0f}"))
    print(f"\n=== RELATÓRIO DE MÉTRICAS ===")
    print(classification_report(y_real_binario, y_pred_binario, labels=labels_bin, zero_division=0))

    return y_real_binario, y_pred_binario, cm

In [8]:
CLASS_ALIASES_LATEX = {'BENIGN': 'BENIGN', 'ATTACK': 'ATTACK'}


def escape_latex(value):
    replacements = {
        "\\": "\\textbackslash{}",
        "&": "\\&",
        "%": "\\%",
        "$": "\\$",
        "#": "\\#",
        "_": "\\_",
        "{": "\\{",
        "}": "\\}",
        "~": "\\textasciitilde{}",
        "^": "\\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in str(value))


def format_confusion_value(value, is_diagonal):
    value = int(value)
    if is_diagonal:
        return f"\\ok{{{value}}}"
    if value != 0:
        return f"\\err{{{value}}}"
    return "0"


def make_latex_confusion_matrix(cm_values, class_labels, caption, table_label):
    headers = [escape_latex(CLASS_ALIASES_LATEX.get(l, l)) for l in class_labels]
    rows = []
    for i, real_label in enumerate(class_labels):
        row_values = [format_confusion_value(cm_values[i][j], i == j) for j in range(len(class_labels))]
        rows.append((f"Real\\_{escape_latex(CLASS_ALIASES_LATEX.get(real_label, real_label))}", row_values))

    first_col_width = max([0] + [len(row_name) for row_name, _ in rows])
    col_widths = [max(len(headers[i]), *(len(values[i]) for _, values in rows)) for i in range(len(headers))]

    def format_row(first_cell, values):
        first = first_cell.ljust(first_col_width)
        rest = " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values))
        return f"            {first} & {rest} \\\\"

    lines = [
        "\\begin{table}[H]",
        "    \\centering",
        "    \\small",
        f"        \\begin{{tabular}}{{l|{'r' * len(class_labels)}}}",
        "            \\hline",
        format_row("", headers),
        "            \\hline",
    ]
    lines.extend(format_row(row_name, row_values) for row_name, row_values in rows)
    lines.extend([
        "            \\hline",
        "        \\end{tabular}",
        "    }",
        f"    \\caption{{{escape_latex(caption)}}}",
        f"    \\label{{{table_label}}}",
        "\\end{table}",
    ])
    return "\n".join(lines)


def format_metric(value):
    return "-" if value is None else f"{value:.2f}"


def make_latex_metrics_report(y_true_values, y_pred_values, class_labels, caption, table_label):
    report = classification_report(
        y_true_values, y_pred_values,
        labels=class_labels, output_dict=True, zero_division=0,
    )
    total_support = int(sum(report[label]["support"] for label in class_labels))
    rows = []
    for label in class_labels:
        metrics = report[label]
        rows.append([
            escape_latex(label),
            format_metric(metrics["precision"]),
            format_metric(metrics["recall"]),
            format_metric(metrics["f1-score"]),
            str(int(metrics["support"])),
        ])

    rows.extend([
        ["\\textbf{Acurácia}", "-", format_metric(report["accuracy"]), "-", str(total_support)],
        ["\\textbf{Média Macro}", format_metric(report["macro avg"]["precision"]), format_metric(report["macro avg"]["recall"]), format_metric(report["macro avg"]["f1-score"]), str(total_support)],
        ["\\textbf{Média Ponderada}", format_metric(report["weighted avg"]["precision"]), format_metric(report["weighted avg"]["recall"]), format_metric(report["weighted avg"]["f1-score"]), str(total_support)],
    ])

    headers = ["Classe", "Precisão", "Revocação", "F1-score", "Suporte"]
    col_widths = [max(len(str(row[i])) for row in [headers] + rows) for i in range(len(headers))]

    def format_row(values):
        return "        " + " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values)) + " \\\\"

    lines = [
        "\\begin{table}[H]",
        "    \\centering",
        "    \\small",
        "    \\begin{tabular}{lrrrr}",
        "        \\hline",
        format_row(headers),
        "        \\hline",
    ]
    lines.extend(format_row(row) for row in rows[:len(class_labels)])
    lines.extend([
        "        \\hline",
        format_row(rows[-3]),
        format_row(rows[-2]),
        format_row(rows[-1]),
        "        \\hline",
        "    \\end{tabular}",
        f"    \\caption{{{escape_latex(caption)}}}",
        f"    \\label{{{table_label}}}",
        "\\end{table}",
    ])
    return "\n".join(lines)

In [9]:
df_teste = pd.read_csv(caminho_pasta_tratado + nome_dados_teste)
y_real, y_pred, cm = avaliar_autoencoder(autoencoder, limiar, df_teste, 'Teste Completo', 'benign')

labels_bin = ['BENIGN', 'ATTACK']

print(make_latex_confusion_matrix(
    cm, labels_bin,
    'Autoencoder — LycoS-IDS2017 (Teste Completo) — Matriz de Confusão',
    'table:ae_lycos_completo_mc',
))
print()
print(make_latex_metrics_report(
    y_real, y_pred, labels_bin,
    'Autoencoder — LycoS-IDS2017 (Teste Completo) — Relatório de Métricas',
    'table:ae_lycos_completo_metricas',
))


CENÁRIO: Teste Completo
Total de amostras: 551250
  BENIGN: 418639
  ATTACK: 132611

=== MATRIZ DE CONFUSÃO ===


,Pred_BENIGN,Pred_ATTACK
Real_BENIGN,397515,21124
Real_ATTACK,23466,109145



=== RELATÓRIO DE MÉTRICAS ===
              precision    recall  f1-score   support

      BENIGN       0.94      0.95      0.95    418639
      ATTACK       0.84      0.82      0.83    132611

    accuracy                           0.92    551250
   macro avg       0.89      0.89      0.89    551250
weighted avg       0.92      0.92      0.92    551250

\begin{table}[H]
    \centering
    \small
        \begin{tabular}{l|rr}
            \hline
                         & BENIGN      & ATTACK      \\
            \hline
            Real\_BENIGN & \ok{397515} & \err{21124} \\
            Real\_ATTACK & \err{23466} & \ok{109145} \\
            \hline
        \end{tabular}
    }
    \caption{Autoencoder — LycoS-IDS2017 (Teste Completo) — Matriz de Confusão}
    \label{table:ae_lycos_completo_mc}
\end{table}

\begin{table}[H]
    \centering
    \small
    \begin{tabular}{lrrrr}
        \hline
        Classe                   & Precisão & Revocação & F1-score & Suporte \\
        \hline
    